# AI-Based Sales Forecasting & Customer Intelligence System — Fixed Version

This notebook is a cleaned and working version of the project.

### What it contains
1. Sales-data loading / demo-data fallback
2. Data validation and cleaning
3. Sales KPIs and business analysis
4. Monthly sales trend analysis
5. RFM customer intelligence
6. K-Means customer segmentation
7. Data-driven segment naming
8. Time-based sales forecasting with a proper train/test split
9. Recursive future forecasting
10. Model evaluation
11. Forecast and customer-segment exports
12. Optional Supabase upload

**Important:** Run the notebook from top to bottom. Do not run the old duplicated Supabase/debug cells from the original notebook.


In [ ]:
# 1. Install/import required Python libraries
# Most environments already have these. If a package is missing, install it separately.

import warnings
warnings.filterwarnings("ignore")

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor

print("Libraries loaded successfully!")


## 2. Load the sales dataset

If `sales_data.csv` is present in the notebook folder, it is used.

If it is not present, a reproducible demo dataset is created so the project can still run from start to finish.


In [ ]:
# 2. Load existing data or create a reproducible demo dataset

DATA_FILE = "sales_data.csv"

if os.path.exists(DATA_FILE):
    df = pd.read_csv(DATA_FILE)
    print(f"Loaded existing dataset: {DATA_FILE}")
else:
    np.random.seed(42)

    # 36 months of demo data gives the forecasting model enough history.
    n = 1800

    products = [
        "Laptop", "Smartphone", "Tablet", "Keyboard",
        "Mouse", "Headphones", "Monitor", "Printer"
    ]

    categories = {
        "Laptop": "Electronics",
        "Smartphone": "Electronics",
        "Tablet": "Electronics",
        "Keyboard": "Accessories",
        "Mouse": "Accessories",
        "Headphones": "Accessories",
        "Monitor": "Electronics",
        "Printer": "Office"
    }

    regions = ["Nagpur", "Pune", "Mumbai", "Amravati", "Nashik"]
    payment_modes = ["UPI", "Credit Card", "Debit Card", "Cash", "Net Banking"]

    prices = {
        "Laptop": 55000,
        "Smartphone": 30000,
        "Tablet": 22000,
        "Keyboard": 2500,
        "Mouse": 1200,
        "Headphones": 3500,
        "Monitor": 15000,
        "Printer": 12000
    }

    dates = pd.date_range("2023-01-01", "2025-12-31", periods=n)
    rows = []

    for i, date in enumerate(dates):
        product = np.random.choice(products)
        quantity = np.random.randint(1, 5)
        base_price = prices[product]

        # Mild trend + seasonality makes the demo data meaningful for forecasting.
        month_factor = 1 + 0.12 * np.sin(2 * np.pi * date.month / 12)
        trend_factor = 1 + 0.0004 * i
        noise = np.random.normal(1, 0.08)

        sales = max(100, base_price * quantity * month_factor * trend_factor * noise)
        profit = sales * np.random.uniform(0.08, 0.25)

        rows.append([
            f"ORD{i+1:05d}",
            date,
            f"C{np.random.randint(1, 301):03d}",
            product,
            categories[product],
            quantity,
            round(sales, 2),
            round(profit, 2),
            np.random.choice(regions),
            np.random.choice(payment_modes)
        ])

    df = pd.DataFrame(rows, columns=[
        "Order_ID", "Order_Date", "Customer_ID", "Product",
        "Category", "Quantity", "Sales", "Profit",
        "Region", "Payment_Mode"
    ])

    df.to_csv(DATA_FILE, index=False)
    print("No sales_data.csv found. Reproducible demo dataset created and saved.")

print("Shape:", df.shape)
display(df.head())


In [ ]:
# 3. Validate and clean the dataset

required_columns = [
    "Order_ID", "Order_Date", "Customer_ID", "Product",
    "Category", "Quantity", "Sales", "Profit", "Region", "Payment_Mode"
]

missing_columns = [c for c in required_columns if c not in df.columns]

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

df = df.copy()
df["Order_Date"] = pd.to_datetime(df["Order_Date"], errors="coerce")

numeric_columns = ["Quantity", "Sales", "Profit"]
for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

before = len(df)
df = df.dropna(subset=["Order_ID", "Order_Date", "Customer_ID", "Sales"])
df = df.drop_duplicates(subset=["Order_ID"])

print("Rows before cleaning:", before)
print("Rows after cleaning :", len(df))
print("Missing values:")
display(df.isnull().sum().to_frame("Missing"))


In [ ]:
# 4. Business KPIs

total_sales = df["Sales"].sum()
total_profit = df["Profit"].sum()
total_orders = df["Order_ID"].nunique()
total_customers = df["Customer_ID"].nunique()
average_order_value = df["Sales"].mean()
profit_margin = (total_profit / total_sales * 100) if total_sales else 0

kpis = pd.DataFrame({
    "Metric": [
        "Total Sales", "Total Profit", "Total Orders",
        "Total Customers", "Average Order Value", "Profit Margin %"
    ],
    "Value": [
        total_sales, total_profit, total_orders,
        total_customers, average_order_value, profit_margin
    ]
})

display(kpis)


In [ ]:
# 5. Sales analysis by category, region and product

category_sales = df.groupby("Category", as_index=False)["Sales"].sum().sort_values("Sales", ascending=False)
region_sales = df.groupby("Region", as_index=False)["Sales"].sum().sort_values("Sales", ascending=False)
product_sales = df.groupby("Product", as_index=False)["Sales"].sum().sort_values("Sales", ascending=False)

print("Sales by Category")
display(category_sales)

print("Sales by Region")
display(region_sales)

print("Sales by Product")
display(product_sales)


In [ ]:
# 6. Monthly sales time series

monthly_df = (
    df.set_index("Order_Date")
      .resample("MS")["Sales"]
      .sum()
      .reset_index()
      .rename(columns={"Order_Date": "Month"})
)

monthly_df["Month"] = pd.to_datetime(monthly_df["Month"])
monthly_df = monthly_df.sort_values("Month").reset_index(drop=True)

print("Number of monthly observations:", len(monthly_df))
display(monthly_df.head())

plt.figure(figsize=(12, 5))
plt.plot(monthly_df["Month"], monthly_df["Sales"], marker="o")
plt.title("Monthly Sales Trend")
plt.xlabel("Month")
plt.ylabel("Sales")
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 7. Customer Intelligence — RFM Analysis

RFM means:
- **Recency:** how recently the customer purchased
- **Frequency:** how many orders the customer made
- **Monetary:** how much the customer spent

These features are used for customer segmentation.


In [ ]:
# 7. Build RFM customer features

reference_date = df["Order_Date"].max() + pd.Timedelta(days=1)

rfm = (
    df.groupby("Customer_ID")
      .agg(
          Recency=("Order_Date", lambda x: (reference_date - x.max()).days),
          Frequency=("Order_ID", "nunique"),
          Monetary=("Sales", "sum")
      )
      .reset_index()
)

print("Customers:", len(rfm))
display(rfm.head())
display(rfm[["Recency", "Frequency", "Monetary"]].describe())


In [ ]:
# 8. Scale RFM features

rfm_features = rfm[["Recency", "Frequency", "Monetary"]].copy()

scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm_features)

print("RFM features standardized successfully.")


In [ ]:
# 9. Select a practical number of K-Means clusters using silhouette score

max_k = min(6, len(rfm) - 1)
candidate_k = list(range(2, max_k + 1))

scores = {}

for k in candidate_k:
    km = KMeans(n_clusters=k, random_state=42, n_init=20)
    labels = km.fit_predict(rfm_scaled)
    scores[k] = silhouette_score(rfm_scaled, labels)

silhouette_table = pd.DataFrame({
    "K": list(scores.keys()),
    "Silhouette_Score": list(scores.values())
})

display(silhouette_table)

# Keep 4 clusters when possible for a simple business-friendly project.
# If there are fewer than 4 customers, use the maximum available.
N_CLUSTERS = min(4, len(rfm))

kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=20)
rfm["Cluster"] = kmeans.fit_predict(rfm_scaled)

final_silhouette = silhouette_score(rfm_scaled, rfm["Cluster"]) if N_CLUSTERS > 1 else np.nan

print("Selected clusters:", N_CLUSTERS)
print("Final silhouette score:", round(final_silhouette, 4))


## 8. Give clusters meaningful names

A major problem in the original notebook was assigning names such as `High-Value` or `At-Risk` directly to cluster numbers.

K-Means cluster IDs are arbitrary. Cluster `0` is not automatically the same business segment every time.

The code below names clusters from their actual RFM characteristics instead.


In [ ]:
# 10. Create data-driven customer segment labels

cluster_profile = (
    rfm.groupby("Cluster")[["Recency", "Frequency", "Monetary"]]
       .mean()
       .sort_index()
)

# Rank each cluster on business meaning.
# High monetary/frequency + low recency => valuable/active.
cluster_profile["ValueScore"] = (
    cluster_profile["Monetary"].rank(pct=True)
    + cluster_profile["Frequency"].rank(pct=True)
    + (1 - cluster_profile["Recency"].rank(pct=True))
)

active_cluster = cluster_profile["ValueScore"].idxmax()
at_risk_cluster = cluster_profile["Recency"].idxmax()

remaining = [c for c in cluster_profile.index if c not in [active_cluster, at_risk_cluster]]

if remaining:
    regular_cluster = max(
        remaining,
        key=lambda c: cluster_profile.loc[c, "Frequency"]
    )
else:
    regular_cluster = active_cluster

occasional_candidates = [c for c in cluster_profile.index if c not in [active_cluster, at_risk_cluster, regular_cluster]]
occasional_cluster = occasional_candidates[0] if occasional_candidates else regular_cluster

segment_map = {
    active_cluster: "High-Value Customers",
    at_risk_cluster: "At-Risk Customers",
    regular_cluster: "Regular Customers",
    occasional_cluster: "Occasional Customers"
}

rfm["Customer_Segment"] = rfm["Cluster"].map(segment_map)

segment_summary = (
    rfm.groupby("Customer_Segment")
       .agg(
           Customers=("Customer_ID", "count"),
           Avg_Recency=("Recency", "mean"),
           Avg_Frequency=("Frequency", "mean"),
           Avg_Monetary=("Monetary", "mean")
       )
       .reset_index()
)

print("Cluster profiles:")
display(cluster_profile)

print("Customer segment summary:")
display(segment_summary)


In [ ]:
# 11. Customer segmentation visualization

plt.figure(figsize=(10, 6))

for segment in sorted(rfm["Customer_Segment"].unique()):
    part = rfm[rfm["Customer_Segment"] == segment]
    plt.scatter(part["Frequency"], part["Monetary"], label=segment, alpha=0.7)

plt.title("Customer Segmentation using RFM + K-Means")
plt.xlabel("Purchase Frequency")
plt.ylabel("Total Spending")
plt.legend()
plt.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()


## 9. Sales Forecasting

The original forecasting section had a major evaluation problem: the Random Forest was trained on **all observations and then evaluated on those same observations**. That produces training-set metrics, not a genuine test result.

This version uses a chronological train/test split.

It also uses lag and rolling features, then forecasts future months recursively.


In [ ]:
# 12. Prepare forecasting features

forecast_df = monthly_df.copy()

# Calendar features
forecast_df["Year"] = forecast_df["Month"].dt.year
forecast_df["Month_Number"] = forecast_df["Month"].dt.month
forecast_df["Time_Index"] = np.arange(len(forecast_df))

# Lag features
for lag in [1, 2, 3]:
    forecast_df[f"Lag_{lag}"] = forecast_df["Sales"].shift(lag)

forecast_df["Rolling_Mean_3"] = forecast_df["Sales"].shift(1).rolling(3).mean()

feature_columns = [
    "Year", "Month_Number", "Time_Index",
    "Lag_1", "Lag_2", "Lag_3", "Rolling_Mean_3"
]

model_df = forecast_df.dropna().reset_index(drop=True)

print("Forecasting rows after feature creation:", len(model_df))
display(model_df.head())


In [ ]:
# 13. Chronological train/test split

if len(model_df) < 8:
    raise ValueError(
        "Not enough monthly observations for a reliable train/test forecast. "
        "Please provide at least about 12 months of sales history."
    )

split_index = max(5, int(len(model_df) * 0.80))
if split_index >= len(model_df):
    split_index = len(model_df) - 1

train = model_df.iloc[:split_index].copy()
test = model_df.iloc[split_index:].copy()

X_train = train[feature_columns]
y_train = train["Sales"]

X_test = test[feature_columns]
y_test = test["Sales"]

model = RandomForestRegressor(
    n_estimators=300,
    max_depth=8,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

test["Predicted_Sales"] = model.predict(X_test)

mae = mean_absolute_error(y_test, test["Predicted_Sales"])
rmse = np.sqrt(mean_squared_error(y_test, test["Predicted_Sales"]))
r2 = r2_score(y_test, test["Predicted_Sales"]) if len(test) > 1 else np.nan

print("Train months:", len(train))
print("Test months :", len(test))
print("MAE         :", round(mae, 2))
print("RMSE        :", round(rmse, 2))
print("R²          :", round(r2, 4) if not np.isnan(r2) else "N/A")

display(test[["Month", "Sales", "Predicted_Sales"]])


In [ ]:
# 14. Plot test-set actual vs predicted sales

plt.figure(figsize=(12, 5))
plt.plot(test["Month"], test["Sales"], marker="o", label="Actual Sales")
plt.plot(test["Month"], test["Predicted_Sales"], marker="o", label="Predicted Sales")
plt.title("Test Set: Actual vs Predicted Sales")
plt.xlabel("Month")
plt.ylabel("Sales")
plt.xticks(rotation=45)
plt.legend()
plt.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()


In [ ]:
# 15. Train final model on all available supervised observations

final_model = RandomForestRegressor(
    n_estimators=300,
    max_depth=8,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

final_model.fit(model_df[feature_columns], model_df["Sales"])

print("Final forecasting model trained on all available historical training rows.")


In [ ]:
# 16. Recursive future forecasting

FORECAST_HORIZON = 3

history = monthly_df[["Month", "Sales"]].copy()
future_rows = []

for step in range(FORECAST_HORIZON):
    next_month = history["Month"].max() + pd.offsets.MonthBegin(1)

    lag_1 = history["Sales"].iloc[-1]
    lag_2 = history["Sales"].iloc[-2] if len(history) >= 2 else lag_1
    lag_3 = history["Sales"].iloc[-3] if len(history) >= 3 else lag_2
    rolling_mean_3 = np.mean([lag_1, lag_2, lag_3])

    row = pd.DataFrame([{
        "Year": next_month.year,
        "Month_Number": next_month.month,
        "Time_Index": len(history),
        "Lag_1": lag_1,
        "Lag_2": lag_2,
        "Lag_3": lag_3,
        "Rolling_Mean_3": rolling_mean_3
    }])

    prediction = float(final_model.predict(row[feature_columns])[0])

    future_rows.append({
        "Month": next_month,
        "Predicted_Sales": max(0, prediction)
    })

    history = pd.concat([
        history,
        pd.DataFrame([{"Month": next_month, "Sales": max(0, prediction)}])
    ], ignore_index=True)

future_forecast = pd.DataFrame(future_rows)

print("Future sales forecast:")
display(future_forecast)


In [ ]:
# 17. Feature importance

importance = (
    pd.DataFrame({
        "Feature": feature_columns,
        "Importance": final_model.feature_importances_
    })
    .sort_values("Importance", ascending=False)
)

display(importance)

plt.figure(figsize=(9, 5))
plt.barh(importance["Feature"], importance["Importance"])
plt.gca().invert_yaxis()
plt.title("Random Forest Feature Importance")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()


In [ ]:
# 18. Save project outputs

monthly_df.to_csv("monthly_sales.csv", index=False)
rfm.to_csv("customer_segments.csv", index=False)
segment_summary.to_csv("customer_segment_summary.csv", index=False)
future_forecast.to_csv("future_sales_forecast.csv", index=False)
test.to_csv("forecast_test_results.csv", index=False)

metrics = pd.DataFrame({
    "Metric": ["MAE", "RMSE", "R2", "Silhouette Score"],
    "Value": [mae, rmse, r2, final_silhouette]
})
metrics.to_csv("model_metrics.csv", index=False)

print("Project output files created:")
for f in [
    "monthly_sales.csv",
    "customer_segments.csv",
    "customer_segment_summary.csv",
    "future_sales_forecast.csv",
    "forecast_test_results.csv",
    "model_metrics.csv"
]:
    print(" -", f)


## 10. Optional: Supabase integration

Do this **only after the notebook works locally/inside Colab**.

Do not put a Supabase secret/service-role key directly into a notebook that you will share publicly. Use an environment variable or a secure secret store.

Expected tables can be:

### `sales_predictions`
- `prediction_date`
- `predicted_sales`
- `model_name`

### `customer_segments`
- `customer_id`
- `recency`
- `frequency`
- `monetary`
- `cluster`
- `segment_name`

If your existing Supabase table uses different column names, change the upload mapping instead of repeatedly changing the ML code.


In [ ]:
# 19. OPTIONAL Supabase upload
# Uncomment and configure only after the ML project works.

# %pip install -q supabase

# import os
# from supabase import create_client

# SUPABASE_URL = os.environ["SUPABASE_URL"]
# SUPABASE_KEY = os.environ["SUPABASE_KEY"]

# supabase = create_client(SUPABASE_URL, SUPABASE_KEY)

# prediction_records = [
#     {
#         "prediction_date": row["Month"].strftime("%Y-%m-%d"),
#         "predicted_sales": float(row["Predicted_Sales"]),
#         "model_name": "Random Forest Regressor"
#     }
#     for _, row in future_forecast.iterrows()
# ]

# customer_records = (
#     rfm.rename(columns={
#         "Customer_ID": "customer_id",
#         "Recency": "recency",
#         "Frequency": "frequency",
#         "Monetary": "monetary",
#         "Cluster": "cluster",
#         "Customer_Segment": "segment_name"
#     })[
#         ["customer_id", "recency", "frequency", "monetary", "cluster", "segment_name"]
#     ]
#     .to_dict(orient="records")
# )

# supabase.table("sales_predictions").upsert(
#     prediction_records,
#     on_conflict="prediction_date"
# ).execute()

# supabase.table("customer_segments").upsert(
#     customer_records,
#     on_conflict="customer_id"
# ).execute()

# print("Supabase upload completed.")


# Final project flow

**Sales Data → Cleaning → KPI Analysis → Monthly Aggregation → Forecasting Model → Future Forecast**

**Sales Data → Customer Transactions → RFM → Scaling → K-Means → Customer Segments → Customer Intelligence**

### Suggested project title
**AI-Based Sales Forecasting and Customer Intelligence System Using Machine Learning**

### Important limitation
If you replace the demo data with real sales data, the quality of the forecast depends heavily on having enough historical monthly observations and meaningful sales patterns. The model metrics shown in this notebook are not proof of real-world future accuracy; they are evaluation results on the supplied test period.
